##### Download & Tiền xử lý FAIDSet + MAGE

## Phần 1 — Download & Tổ chức raw data
- FAIDSet (`ngocminhta/FAIDSet`): giữ nguyên split gốc train / valid / test
- MAGE train (`yaful/MAGE`): 15,000 mẫu trộn vào train EN
- MAGE test (`yaful/MAGE`): toàn bộ mẫu cho test EN

```
raw_data/
├── train/      eng/{AI,human}/  vi/{AI,human}/
├── valid/      eng/{AI,human}/  vi/{AI,human}/
├── test/       eng/{AI,human}/  vi/{AI,human}/
├── mage_test/  {AI,human}/   ← test EN
└── mage_train/ {AI,human}/   ← trộn vào train EN
```

## Phần 2 — ELT Pipeline
```
E  Extract:   gộp FAIDSet train+valid + MAGE train → train_all EN
              gộp FAIDSet test EN + MAGE test → test_en
              FAIDSet test VI → test_vi
L  Load:      đọc .txt → DataFrame
T  Transform: clean, filter, trích 10% val → train.csv / val.csv / test_en.csv / test_vi.csv
```

In [7]:
!pip install huggingface_hub langdetect datasets pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 39.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


---
## Phần 1 — Download & Tổ chức raw data

In [8]:
import json, shutil
from pathlib import Path
from collections import Counter
from huggingface_hub import hf_hub_download
from langdetect import detect, LangDetectException

BASE = Path("raw_data")
TMP  = Path("tmp_download")
TMP.mkdir(exist_ok=True)

for split in ["train", "valid", "test"]:
    for d in ["eng/AI", "eng/human", "vi/AI", "vi/human"]:
        (BASE / split / d).mkdir(parents=True, exist_ok=True)

SPLITS = {"train": "train.jsonl", "valid": "valid.jsonl", "test": "test.jsonl"}
raw_files = {}
for split, filename in SPLITS.items():
    path = hf_hub_download(
        repo_id="ngocminhta/FAIDSet",
        filename=filename,
        repo_type="dataset",
        local_dir=str(TMP)
    )
    raw_files[split] = Path(path)
    print(f"Downloaded {split}: {path}")

def get_kind(record):
    label = str(record.get("label", "")).lower()
    label = label.replace("\u2013", "-").replace("\u2014", "-")
    if label == "human-written": return "human"
    if label == "llm-generated":  return "AI"
    return None  # collaborative → bỏ qua

def get_lang(text):
    try:
        lang = detect(text[:500])
        return {"vi": "vi", "en": "eng"}.get(lang)
    except LangDetectException:
        return None

total_skipped = Counter()
for split, path in raw_files.items():
    counts, skipped = Counter(), Counter()
    with open(path, "r", encoding="utf-8") as f:
        records = [json.loads(l) for l in f if l.strip()]
    print(f"\n[{split.upper()}] {len(records):,} records")
    for idx, record in enumerate(records):
        kind = get_kind(record)
        if not kind:  skipped["collaborative"] += 1; continue
        text = record.get("text", "").strip()
        if not text:  skipped["empty"] += 1;         continue
        lang = get_lang(text)
        if not lang:  skipped["other_lang"] += 1;    continue
        fname = f"{split}_{idx:06d}.txt"
        (BASE / split / lang / kind / fname).write_text(text, encoding="utf-8")
        counts[(lang, kind)] += 1
    for (lang, kind), c in sorted(counts.items()):
        print(f"  raw_data/{split}/{lang}/{kind}/ → {c:,} files")
    print(f"  Bỏ qua: {dict(skipped)}")
    total_skipped += skipped

shutil.rmtree(TMP)
print(f"\n Xong FAIDSet! Tổng bỏ qua: {dict(total_skipped)}")


train.jsonl:   0%|          | 0.00/49.6M [00:00<?, ?B/s]

Downloaded train: tmp_download/train.jsonl


valid.jsonl:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Downloaded valid: tmp_download/valid.jsonl


test.jsonl:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

Downloaded test: tmp_download/test.jsonl

[TRAIN] 60,676 records
  raw_data/train/eng/AI/ → 7,883 files
  raw_data/train/eng/human/ → 5,052 files
  raw_data/train/vi/AI/ → 6,525 files
  raw_data/train/vi/human/ → 9,110 files
  Bỏ qua: {'collaborative': 32091, 'other_lang': 15}

[VALID] 12,502 records
  raw_data/valid/eng/AI/ → 1,272 files
  raw_data/valid/eng/human/ → 1,039 files
  raw_data/valid/vi/AI/ → 1,316 files
  raw_data/valid/vi/human/ → 1,997 files
  Bỏ qua: {'collaborative': 6876, 'other_lang': 2}

[TEST] 12,505 records
  raw_data/test/eng/AI/ → 1,268 files
  raw_data/test/eng/human/ → 1,074 files
  raw_data/test/vi/AI/ → 1,320 files
  raw_data/test/vi/human/ → 1,959 files
  Bỏ qua: {'collaborative': 6879, 'other_lang': 5}

 Xong FAIDSet! Tổng bỏ qua: {'collaborative': 45846, 'other_lang': 22}


In [9]:
# Download MAGE test → toàn bộ cho test EN
# Download MAGE train → 50,000 mẫu (25,000 human + 25,000 AI) trộn vào train EN
from datasets import load_dataset
import pandas as pd

def save_to_raw(df_in, base_dir, prefix):
    """Lưu DataFrame vào raw_data/{base_dir}/AI/ và /human/."""
    d = BASE / base_dir
    (d / "AI").mkdir(parents=True, exist_ok=True)
    (d / "human").mkdir(parents=True, exist_ok=True)
    for idx, row in df_in.iterrows():
        kind = "human" if row["label"] == 0 else "AI"
        (d / kind / f"{prefix}_{idx:05d}.txt").write_text(row["text"], encoding="utf-8")

def load_mage_split(split_name):
    print(f"Đang load MAGE {split_name}...")
    ds  = load_dataset("yaful/MAGE", split=split_name)
    df  = ds.to_pandas()[["text", "label"]].copy()
    # MAGE: label=0 machine-generated, label=1 human → đổi: 0=human, 1=AI
    df["label"] = df["label"].apply(lambda x: 0 if x == 1 else 1)
    df = df[df["text"].str.len() >= 50].reset_index(drop=True)
    return df

# ── MAGE test: lấy toàn bộ ──────────────────────────────────────────
df_mage_test = load_mage_split("test")
save_to_raw(df_mage_test, "mage_test", "magetest")
print(f"MAGE test: {len(df_mage_test):,} mẫu")
print(f"   mage_test/AI/    → {len(list((BASE/'mage_test'/'AI').glob('*.txt'))):,} files")
print(f"   mage_test/human/ → {len(list((BASE/'mage_test'/'human').glob('*.txt'))):,} files")

# ── MAGE train: lấy 25,000 human + 25,000 AI ──────────────────────────
df_mage_train = load_mage_split("train")
human_sample = df_mage_train[df_mage_train["label"] == 0].sample(7500, random_state=42)
ai_sample    = df_mage_train[df_mage_train["label"] == 1].sample(7500, random_state=42)
df_mage_train_sample = pd.concat([human_sample, ai_sample]).reset_index(drop=True)
save_to_raw(df_mage_train_sample, "mage_train", "magetrain")
print(f"\nMAGE train sample: {len(df_mage_train_sample):,} mẫu")
print(f"   mage_train/AI/    → {len(list((BASE/'mage_train'/'AI').glob('*.txt'))):,} files")
print(f"   mage_train/human/ → {len(list((BASE/'mage_train'/'human').glob('*.txt'))):,} files")


Đang load MAGE test...


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/404M [00:00<?, ?B/s]

valid.csv:   0%|          | 0.00/72.3M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/71.7M [00:00<?, ?B/s]

...

test_ood_set_gpt_para.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/319071 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/56792 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60743 [00:00<?, ? examples/s]

MAGE test: 60,279 mẫu
   mage_test/AI/    → 30,454 files
   mage_test/human/ → 29,825 files
Đang load MAGE train...

MAGE train sample: 15,000 mẫu
   mage_train/AI/    → 7,500 files
   mage_train/human/ → 7,500 files


In [10]:
print("\n=== Thống kê raw_data ===")
for split in ["train", "valid", "test"]:
    print(f"\n[{split.upper()}]")
    for lang in ["eng", "vi"]:
        for kind in ["AI", "human"]:
            d = BASE / split / lang / kind
            n = len(list(d.glob("*.txt"))) if d.exists() else 0
            print(f"  {split}/{lang}/{kind}/ → {n:,} files")

for folder, label in [("mage_test", "MAGE TEST"), ("mage_train", "MAGE TRAIN")]:
    print(f"\n[{label}]")
    for kind in ["AI", "human"]:
        d = BASE / folder / kind
        n = len(list(d.glob("*.txt"))) if d.exists() else 0
        print(f"  {folder}/{kind}/ → {n:,} files")



=== Thống kê raw_data ===

[TRAIN]
  train/eng/AI/ → 7,883 files
  train/eng/human/ → 5,052 files
  train/vi/AI/ → 6,525 files
  train/vi/human/ → 9,110 files

[VALID]
  valid/eng/AI/ → 1,272 files
  valid/eng/human/ → 1,039 files
  valid/vi/AI/ → 1,316 files
  valid/vi/human/ → 1,997 files

[TEST]
  test/eng/AI/ → 1,268 files
  test/eng/human/ → 1,074 files
  test/vi/AI/ → 1,320 files
  test/vi/human/ → 1,959 files

[MAGE TEST]
  mage_test/AI/ → 30,454 files
  mage_test/human/ → 29,825 files

[MAGE TRAIN]
  mage_train/AI/ → 7,500 files
  mage_train/human/ → 7,500 files


---
## Phần 2 — ELT Pipeline

### E — Extract
Gộp raw files thành các nhóm:
- `train`: FAIDSet train + FAIDSet valid (gộp hết để tối đa data train)
- `test_en`: FAIDSet test EN + MAGE 10k
- `test_vi`: FAIDSet test VI

> Val nội bộ sẽ được trích 10% từ train ở bước Transform.

In [11]:
import shutil
from pathlib import Path

BASE = Path("raw_data")

for kind in ["AI", "human"]:
    (BASE / "train_all" / "eng" / kind).mkdir(parents=True, exist_ok=True)
    (BASE / "train_all" / "vi"  / kind).mkdir(parents=True, exist_ok=True)
    (BASE / "test_en"   / kind).mkdir(parents=True, exist_ok=True)
    (BASE / "test_vi"   / kind).mkdir(parents=True, exist_ok=True)

# train_all EN = FAIDSet train EN + FAIDSet valid EN + MAGE train 15k
# train_all VI = FAIDSet train VI + FAIDSet valid VI
for split in ["train", "valid"]:
    for lang in ["eng", "vi"]:
        for kind in ["AI", "human"]:
            src = BASE / split / lang / kind
            if not src.exists(): continue
            for f in src.glob("*.txt"):
                shutil.copy2(f, BASE / "train_all" / lang / kind / f"{split}_{f.name}")

# MAGE train → train_all/eng/
for kind in ["AI", "human"]:
    src = BASE / "mage_train" / kind
    if not src.exists(): continue
    for f in src.glob("*.txt"):
        shutil.copy2(f, BASE / "train_all" / "eng" / kind / f.name)

# test_en = FAIDSet test EN + MAGE test
for kind in ["AI", "human"]:
    src = BASE / "test" / "eng" / kind
    if src.exists():
        for f in src.glob("*.txt"):
            shutil.copy2(f, BASE / "test_en" / kind / f.name)
    src = BASE / "mage_test" / kind
    if src.exists():
        for f in src.glob("*.txt"):
            shutil.copy2(f, BASE / "test_en" / kind / f.name)

# test_vi = FAIDSet test VI
for kind in ["AI", "human"]:
    src = BASE / "test" / "vi" / kind
    if not src.exists(): continue
    for f in src.glob("*.txt"):
        shutil.copy2(f, BASE / "test_vi" / kind / f.name)

print("=== Kết quả Extract ===")
print("\n[TRAIN_ALL]")
for lang in ["eng", "vi"]:
    for kind in ["AI", "human"]:
        d = BASE / "train_all" / lang / kind
        print(f"  train_all/{lang}/{kind}/ → {len(list(d.glob('*.txt'))):,} files")

for folder, label in [("test_en", "TEST EN"), ("test_vi", "TEST VI")]:
    print(f"\n[{label}]")
    for kind in ["AI", "human"]:
        d = BASE / folder / kind
        print(f"  {folder}/{kind}/ → {len(list(d.glob('*.txt'))):,} files")


=== Kết quả Extract ===

[TRAIN_ALL]
  train_all/eng/AI/ → 16,655 files
  train_all/eng/human/ → 13,591 files
  train_all/vi/AI/ → 7,841 files
  train_all/vi/human/ → 11,107 files

[TEST EN]
  test_en/AI/ → 31,722 files
  test_en/human/ → 30,899 files

[TEST VI]
  test_vi/AI/ → 1,320 files
  test_vi/human/ → 1,959 files


### L — Load
Đọc toàn bộ file `.txt` → DataFrame.

In [12]:
import pandas as pd
from pathlib import Path

BASE      = Path("raw_data")
PROCESSED = Path("processed_data")
PROCESSED.mkdir(exist_ok=True)

FOLDER_MAP_2D = {
    ("eng", "AI"):    {"language": "en", "label": 1},
    ("eng", "human"): {"language": "en", "label": 0},
    ("vi",  "AI"):    {"language": "vi", "label": 1},
    ("vi",  "human"): {"language": "vi", "label": 0},
}

def load_2d(split_name):
    """Đọc từ raw_data/{split}/lang/kind/ → DataFrame."""
    records = []
    for (lang, kind), meta in FOLDER_MAP_2D.items():
        folder = BASE / split_name / lang / kind
        if not folder.exists(): continue
        files = list(folder.glob("*.txt"))
        for f in files:
            records.append({"text": f.read_text(encoding="utf-8"), **meta})
        print(f"  {split_name}/{lang}/{kind}/ → {len(files):,} files")
    return pd.DataFrame(records)

def load_flat(folder_name, language):
    """Đọc từ raw_data/{folder}/kind/ (flat) → DataFrame."""
    records = []
    for kind, label in [("AI", 1), ("human", 0)]:
        folder = BASE / folder_name / kind
        if not folder.exists(): continue
        files = list(folder.glob("*.txt"))
        for f in files:
            records.append({"text": f.read_text(encoding="utf-8"),
                            "language": language, "label": label})
        print(f"  {folder_name}/{kind}/ → {len(files):,} files")
    return pd.DataFrame(records)

print("=== TRAIN ALL ===")
df_train_raw = load_2d("train_all")
print(f"Tổng: {len(df_train_raw):,}\n")

print("=== TEST EN ===")
df_test_en_raw = load_flat("test_en", "en")
print(f"Tổng: {len(df_test_en_raw):,}\n")

print("=== TEST VI ===")
df_test_vi_raw = load_flat("test_vi", "vi")
print(f"Tổng: {len(df_test_vi_raw):,}")


=== TRAIN ALL ===
  train_all/eng/AI/ → 16,655 files
  train_all/eng/human/ → 13,591 files
  train_all/vi/AI/ → 7,841 files
  train_all/vi/human/ → 11,107 files
Tổng: 49,194

=== TEST EN ===
  test_en/AI/ → 31,722 files
  test_en/human/ → 30,899 files
Tổng: 62,621

=== TEST VI ===
  test_vi/AI/ → 1,320 files
  test_vi/human/ → 1,959 files
Tổng: 3,279


### T — Transform
Làm sạch text, filter quá ngắn, trích 10% val nội bộ, lưu CSV.

In [13]:
import re
from sklearn.model_selection import train_test_split

MIN_CHARS = 50

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\S\n]+", " ", text)
    return text.strip()

def transform(df, name):
    df = df.copy()
    df["text"] = df["text"].apply(clean_text)
    before = len(df)
    df = df[df["text"].str.len() >= MIN_CHARS].reset_index(drop=True)
    after = len(df)
    print(f"[{name}] Filter < {MIN_CHARS} ký tự: bỏ {before-after} rows → còn {after:,}")
    print(df.groupby(["language", "label"]).size().to_string())
    print()
    return df

df_train_all = transform(df_train_raw,    "TRAIN ALL")
df_test_en   = transform(df_test_en_raw,  "TEST EN")
df_test_vi   = transform(df_test_vi_raw,  "TEST VI")

# Trích 10% train_all làm val nội bộ, stratify theo language+label
df_train_all["stratify_key"] = df_train_all["language"] + "_" + df_train_all["label"].astype(str)
df_train, df_val = train_test_split(
    df_train_all, test_size=0.1, random_state=42,
    stratify=df_train_all["stratify_key"]
)
df_train = df_train.drop(columns=["stratify_key"]).reset_index(drop=True)
df_val   = df_val.drop(columns=["stratify_key"]).reset_index(drop=True)

df_train.to_csv   (PROCESSED / "train.csv",   index=False, encoding="utf-8-sig")
df_val.to_csv     (PROCESSED / "val.csv",     index=False, encoding="utf-8-sig")
df_test_en.to_csv (PROCESSED / "test_en.csv", index=False, encoding="utf-8-sig")
df_test_vi.to_csv (PROCESSED / "test_vi.csv", index=False, encoding="utf-8-sig")

print(" Đã lưu:")
print(f"   train.csv   → {len(df_train):,} rows")
print(f"   val.csv     → {len(df_val):,} rows  (10% trích từ train)")
print(f"   test_en.csv → {len(df_test_en):,} rows")
print(f"   test_vi.csv → {len(df_test_vi):,} rows")


[TRAIN ALL] Filter < 50 ký tự: bỏ 0 rows → còn 49,194
language  label
en        0        13591
          1        16655
vi        0        11107
          1         7841

[TEST EN] Filter < 50 ký tự: bỏ 0 rows → còn 62,621
language  label
en        0        30899
          1        31722

[TEST VI] Filter < 50 ký tự: bỏ 1 rows → còn 3,278
language  label
vi        0        1958
          1        1320

 Đã lưu:
   train.csv   → 44,274 rows
   val.csv     → 4,920 rows  (10% trích từ train)
   test_en.csv → 62,621 rows
   test_vi.csv → 3,278 rows


In [14]:
# Upload processed_data lên Kaggle Dataset
import json as _json, os, subprocess

dataset_name = "faidset-processed"
kaggle_user  = subprocess.run("kaggle config view", shell=True,
                              capture_output=True, text=True).stdout
kaggle_user  = [l.split(":")[1].strip() for l in kaggle_user.split("\n")
                if "username" in l][0]

upload_dir = "processed_data"
with open(f"{upload_dir}/dataset-metadata.json", "w") as f:
    _json.dump({
        "title"   : dataset_name,
        "id"      : f"{kaggle_user}/{dataset_name}",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

check = subprocess.run(f"kaggle datasets list --user {kaggle_user} --search {dataset_name}",
                       shell=True, capture_output=True, text=True)
if dataset_name in check.stdout:
    result = subprocess.run(f'kaggle datasets version -p {upload_dir} -m "auto update"',
                            shell=True, capture_output=True, text=True)
else:
    result = subprocess.run(f"kaggle datasets create -p {upload_dir}",
                            shell=True, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)
print(f"Done! {kaggle_user}/{dataset_name}")


Starting upload for file test_en.csv
Upload successful: test_en.csv (75MB)
Starting upload for file train.csv
Upload successful: train.csv (44MB)
Starting upload for file test_vi.csv
Upload successful: test_vi.csv (3MB)
Starting upload for file val.csv
Upload successful: val.csv (5MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/faidset-processed


  0%|          | 0.00/75.2M [00:00<?, ?B/s]
  7%|▋         | 5.58M/75.2M [00:00<00:01, 58.5MB/s]
 20%|█▉        | 15.0M/75.2M [00:00<00:00, 72.5MB/s]
 31%|███       | 23.4M/75.2M [00:00<00:00, 75.8MB/s]
 41%|████      | 30.6M/75.2M [00:00<00:00, 49.7MB/s]
 52%|█████▏    | 39.0M/75.2M [00:00<00:00, 52.8MB/s]
 59%|█████▉    | 44.5M/75.2M [00:00<00:00, 48.7MB/s]
 66%|██████▌   | 49.5M/75.2M [00:00<00:00, 48.5MB/s]
 78%|███████▊  | 58.9M/75.2M [00:01<00:00, 58.9MB/s]
 86%|████████▌ | 64.8M/75.2M [00:01<00:00, 48.4MB/s]
 94%|█████████▍| 70.9M/75.2M [00:01<00:00, 51.4MB/s]
100%|██████████| 75